# 09 — Experimento: remover as features mais "delatoras" do shift
## PRT Seguros

No notebook `08_validacao_adversarial.ipynb` vimos que dá pra diferenciar treino de Kaggle com
AUC = 0.72, e que as 10 features mais responsáveis por isso são, em ordem: `indice_relacionamento`,
`tempo_medio_resposta_dias`, `tempo_cliente_dias`, `ano_veiculo`, `km_anual_estimado`,
`desconto_aplicado_pct`, `renda_anual`, `valor_premio_anual`, `valor_imovel`, `score_engajamento_digital`.

**O problema:** várias dessas são também as features **mais preditivas de churn** que existem na base
(`tempo_cliente_dias` é a #1 de todas, correlação 0.298). Simplesmente removê-las tende a piorar mais
do que ajudar. Este notebook testa isso de forma controlada, comparando o CatBoost:

1. Com todas as features (baseline atual)
2. Sem as 10 features mais "delatoras"
3. Sem as 5 features "delatoras" que são fracamente preditivas de churn (mantendo as fortes)

E decide, com base no AUC de validação, se vale a pena remover alguma.

## 1. Carregar dados e checar correlação de cada feature suspeita com o churn

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train = pd.read_csv("dados_processados/train_model_ready.csv")
val = pd.read_csv("dados_processados/val_model_ready.csv")

ID_COL, TARGET_COL = "cod_individuo", "churned"
feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]

TOP10_SHIFT = [
    "indice_relacionamento", "tempo_medio_resposta_dias", "tempo_cliente_dias",
    "ano_veiculo", "km_anual_estimado", "desconto_aplicado_pct", "renda_anual",
    "valor_premio_anual", "valor_imovel", "score_engajamento_digital",
]

corr_churn = train[TOP10_SHIFT + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
corr_churn = corr_churn.reindex(corr_churn.abs().sort_values(ascending=False).index)
print("Correlação de cada feature 'delatora' com o churn (quanto maior |corr|, mais arriscado remover):")
print(corr_churn.round(3))


Correlação de cada feature 'delatora' com o churn (quanto maior |corr|, mais arriscado remover):
tempo_cliente_dias          -0.288
desconto_aplicado_pct        0.222
indice_relacionamento       -0.185
renda_anual                 -0.098
valor_imovel                -0.094
score_engajamento_digital    0.050
valor_premio_anual           0.011
km_anual_estimado            0.005
tempo_medio_resposta_dias    0.004
ano_veiculo                  0.002
Name: churned, dtype: float64


## 2. Definir os conjuntos de features a testar

- **Fortes** (|corr| > 0.05): `tempo_cliente_dias`, `desconto_aplicado_pct`, `indice_relacionamento` —
  mantemos essas mesmo estando na lista de shift, o custo de perder sinal é alto demais.
- **Fracas** (|corr| < 0.05): as demais 7 — candidatas reais a remover, já que contribuem pouco pra
  prever churn mas bastante para o modelo "aprender a reconhecer" se a linha é do treino ou do Kaggle.

In [2]:
FORTES = corr_churn[corr_churn.abs() > 0.05].index.tolist()
FRACAS = corr_churn[corr_churn.abs() <= 0.05].index.tolist()
print(f"Mantemos (correlação forte com churn): {FORTES}")
print(f"Candidatas a remover (correlação fraca, mas shift alto): {FRACAS}")

conjuntos = {
    "baseline_todas_features": feature_cols,
    "sem_top10_shift": [c for c in feature_cols if c not in TOP10_SHIFT],
    "sem_fracas_shift": [c for c in feature_cols if c not in FRACAS],
}


Mantemos (correlação forte com churn): ['tempo_cliente_dias', 'desconto_aplicado_pct', 'indice_relacionamento', 'renda_anual', 'valor_imovel', 'score_engajamento_digital']
Candidatas a remover (correlação fraca, mas shift alto): ['valor_premio_anual', 'km_anual_estimado', 'tempo_medio_resposta_dias', 'ano_veiculo']


## 3. Treinar CatBoost para cada conjunto de features e comparar AUC-ROC de validação

Mesmos hiperparâmetros do `05_catboost_proba.ipynb`, só mudando as colunas de entrada.

In [3]:
resultados_exp = {}

for nome, cols in conjuntos.items():
    X_train, y_train = train[cols], train[TARGET_COL]
    X_val, y_val = val[cols], val[TARGET_COL]

    X_tr_es, X_es, y_tr_es, y_es = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    neg_pos_ratio = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

    modelo = CatBoostClassifier(
        iterations=2000, depth=6, learning_rate=0.03,
        scale_pos_weight=neg_pos_ratio, eval_metric="AUC",
        random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
    )
    modelo.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))

    proba_val = modelo.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, proba_val)
    resultados_exp[nome] = auc
    print(f"{nome:<28} ({len(cols):>2} features) -> AUC-ROC = {auc:.4f}")


baseline_todas_features      (72 features) -> AUC-ROC = 0.8254


sem_top10_shift              (62 features) -> AUC-ROC = 0.7627


sem_fracas_shift             (68 features) -> AUC-ROC = 0.8255


## 4. Conclusão

Se `sem_fracas_shift` (ou `sem_top10_shift`) tiver AUC de validação **muito próximo** do baseline,
vale a pena usar esse conjunto reduzido no Kaggle — a hipótese é que, mesmo perdendo pouquíssimo
sinal de churn, o modelo fica menos "viciado" em padrões que só existem no treino, o que pode ajudar
mais no público do que a queda de validação sugere. Se a queda for grande, não compensa — o próximo
notebook (`10_feature_engineering.ipynb`) segue usando o conjunto de features vencedor aqui como ponto
de partida.

In [4]:
melhor = max(resultados_exp, key=resultados_exp.get)
print("=== Resumo ===")
for nome, auc in sorted(resultados_exp.items(), key=lambda x: -x[1]):
    delta = auc - resultados_exp["baseline_todas_features"]
    print(f"{nome:<28} AUC={auc:.4f}  (delta vs baseline: {delta:+.4f})")

print(f"\nConjunto escolhido para os próximos notebooks: {melhor}")

import json
with open("dados_processados/features_selecionadas.json", "w") as f:
    json.dump({"conjunto_escolhido": melhor, "colunas": conjuntos[melhor]}, f, indent=2)
print("Salvo em dados_processados/features_selecionadas.json")


=== Resumo ===
sem_fracas_shift             AUC=0.8255  (delta vs baseline: +0.0001)
baseline_todas_features      AUC=0.8254  (delta vs baseline: +0.0000)
sem_top10_shift              AUC=0.7627  (delta vs baseline: -0.0627)

Conjunto escolhido para os próximos notebooks: sem_fracas_shift
Salvo em dados_processados/features_selecionadas.json
